In [3]:
import pandas as pd
import os
from datetime import datetime

# ==========================================
# 1. CONFIGURACIÓN EDITABLE
# ==========================================
METAS = {
    "31-60": 2208249.80,   #
    "61-90": 4184439.54,   #
    "91-120": 1010891.51,  #
    "TOTAL": 7403580.85    #[cite: 1]
}

META_REPOS = 13            #[cite: 1]
REPOS_LOGRADAS = 5  

RUTA_PAGOS = r'C:\Users\SISTEMAS\Documents\MAQUINA DIEGO\CARTERAS\GMF\PAGO IMPORTACION\2026\04 ABRIL'
DIAS_LABORADOS_MES = 24
DIAS_TRANSCURRIDOS = 10 
TOTAL_ASESORES = 15

# ==========================================
# 2. CONSULTA SQL PARA RPC (PARA TU REFERENCIA)
# ==========================================
QUERY_RPC = """
SELECT DISTINCT a.ID_CUENTA, a.ESTATUS, b.[días de mora]
FROM [METAGABSSA_2025].[dbo].[historico_resumen_gestion] a 
LEFT JOIN [BAHIA].[dbo].[MACRO_GMF] b ON a.id_cuenta = b.id_cuenta
WHERE a.Id_Cliente IN ('GMF_PRELEGAL','GMF') AND a.Accion <> 'Gestor Visita'
"""

# ==========================================
# 3. PROCESAMIENTO DE DATOS
# ==========================================
def generar_reporte():
    archivos = [os.path.join(RUTA_PAGOS, f) for f in os.listdir(RUTA_PAGOS) if f.endswith('.xlsx')]
    if not archivos:
        print("No se encontraron archivos Excel.")
        return

    dfs = []
    for f in archivos:
        temp = pd.read_excel(f)
        temp.columns = temp.columns.str.strip().str.lower()
        dfs.append(temp)
    
    df = pd.concat(dfs, ignore_index=True)

    # Identificación de columnas según tu archivo
    col_monto = 'monto de pago'
    col_bucket_archivo = 'bucket < 91'
    col_queue = 'queue'

    # --- LIMPIEZA DE MONTO ---
    # Si el monto viene como "$ 8,605.00" lo convertimos a número puro
    if df[col_monto].dtype == 'object':
        df[col_monto] = df[col_monto].replace({r'\$': '', ',': ''}, regex=True).astype(float)

    # --- FILTROS ---
    # 1. Excluir Q_REPOOPEN
    if col_queue in df.columns:
        df = df[df[col_queue].astype(str).str.upper() != 'Q_REPOOPEN']

    # 2. NO QUITAR DUPLICADOS (Para sumar todas las exhibiciones/abonos)

    # --- CÁLCULOS POR BUCKET ---
    total_recuperado_real = df[col_monto].sum()
    dias_rest = DIAS_LABORADOS_MES - DIAS_TRANSCURRIDOS
    rpc_valor = 11.2 # Este valor lo sacas de tu consulta de gestión[cite: 1]

    filas_html = ""
    for b_name, b_meta in METAS.items():
        if b_name == "TOTAL": continue
        
        # Filtramos por el texto que contiene la columna 'bucket < 91' (ej. "31-60")
        mask = df[col_bucket_archivo].astype(str).str.contains(b_name, na=False)
        recup = df[mask][col_monto].sum()
        
        avance = (recup / b_meta * 100) if b_meta > 0 else 0
        falta = b_meta - recup
        diario_asesor = (falta / dias_rest / TOTAL_ASESORES) if dias_rest > 0 else 0
        
        color_class = "verde" if avance >= (DIAS_TRANSCURRIDOS/DIAS_LABORADOS_MES*100) else "rojo"
        
        filas_html += f"""
        <tr>
            <td>Bucket {b_name}</td>
            <td>${b_meta:,.2f}</td>
            <td class="{color_class}">${recup:,.2f}</td>
            <td>{avance:.1f}%</td>
            <td><b>${diario_asesor:,.2f}</b></td>
        </tr>
        """

    # --- GENERACIÓN DEL HTML ---
    html_template = f"""
    <html>
    <head>
        <meta charset="UTF-8">
        <style>
            body {{ font-family: 'Segoe UI', sans-serif; margin: 30px; background-color: #f4f7f6; }}
            .header {{ background: #003366; color: white; padding: 25px; border-radius: 12px; text-align: center; }}
            .container {{ display: flex; gap: 20px; margin: 25px 0; }}
            .card {{ background: white; padding: 20px; border-radius: 12px; flex: 1; box-shadow: 0 4px 6px rgba(0,0,0,0.05); text-align: center; border-top: 5px solid #003366; }}
            .card p {{ font-size: 26px; font-weight: bold; color: #003366; margin: 10px 0; }}
            table {{ width: 100%; border-collapse: collapse; background: white; border-radius: 12px; overflow: hidden; }}
            th {{ background: #003366; color: white; padding: 18px; }}
            td {{ padding: 15px; border-bottom: 1px solid #dee2e6; text-align: center; }}
            .verde {{ color: #2d6a4f; font-weight: bold; }}
            .rojo {{ color: #a4133c; font-weight: bold; }}
            .progress-bg {{ background: #e9ecef; height: 12px; border-radius: 6px; margin-top: 10px; overflow: hidden; }}
            .progress-fill {{ background: #ffc300; height: 100%; width: {(REPOS_LOGRADAS/META_REPOS*100):.1f}%; }}
        </style>
    </head>
    <body>
        <div class="header">
            <h1>GABSSA - Dashboard de Recuperación GMF</h1>
            <p>Estado al día {DIAS_TRANSCURRIDOS} de {DIAS_LABORADOS_MES}</p>
        </div>
        <div class="container">
            <div class="card"><h2>RPC Diario</h2><p>{rpc_valor}%</p></div>
            <div class="card"><h2>Recuperado Total</h2><p>${total_recuperado_real:,.2f}</p></div>
            <div class="card">
                <h2>Unidades Repo</h2>
                <p>{REPOS_LOGRADAS} / {META_REPOS}</p>
                <div class="progress-bg"><div class="progress-fill"></div></div>
            </div>
        </div>
        <table>
            <thead>
                <tr><th>Segmento</th><th>Meta Mes</th><th>Recuperado Real</th><th>% Avance</th><th>Meta Diaria x Asesor</th></tr>
            </thead>
            <tbody>{filas_html}</tbody>
        </table>
    </body>
    </html>
    """
    
    with open("Dash_Recuperacion_GMF.html", "w", encoding="utf-8") as f:
        f.write(html_template)
    print("Dashboard generado. Se han sumado todos los abonos por contrato.")

if __name__ == "__main__":
    generar_reporte()

Dashboard generado. Se han sumado todos los abonos por contrato.
